In [50]:
# ============================================================
# LANGKAH 1: Import Library & Load Dataset
# ============================================================
import pandas as pd
import numpy as np

# 1. Load Dataset
df = pd.read_csv('output/tabel_perbandingan_lengkap_cosine.csv')
print(f"Total data: {len(df)}")
print(f"Kolom: {list(df.columns)}")
print(f"\nLabel unik: {df['label'].unique()}")
print(f"Jenis unik: {df['jenis'].unique()}")
print(f"\nJumlah per label & jenis:")
print(df.groupby(['label', 'jenis']).size())
print("\n5 data pertama:")
df.head()

Total data: 22
Kolom: ['label', 'jenis', 'teks_dialog', 'teks_kritik', 'skor_critic', 'waktu_detik']

Label unik: <StringArray>
['good', 'bad']
Length: 2, dtype: str
Jenis unik: <StringArray>
['original', 'rephrase']
Length: 2, dtype: str

Jumlah per label & jenis:
label  jenis   
bad    original     1
       rephrase    10
good   original     1
       rephrase    10
dtype: int64

5 data pertama:


,label,jenis,teks_dialog,teks_kritik,skor_critic,waktu_detik
0,good,original,"Kak, pacarku lagi marah banget gara-gara kemar...",Dialog menyampaikan masalah yang jelas dan spe...,4.88,8.162
1,good,rephrase,"Kak, pacarku lagi sangat marah karena kemarin ...",Dialog menyampaikan curhat secara jelas dan em...,4.50,7.614
2,good,rephrase,"Kak, pacarku lagi sangat marah karena kemarin ...",Dialog menyampaikan konflik yang jelas dan spe...,4.50,7.011
3,good,rephrase,"Kak, pacarku lagi sangat marah karena kemarin ...",Dialog menyampaikan konflik yang konkret dan s...,4.50,7.838
4,good,rephrase,"Kak, pacarku lagi sangat marah karena kemarin ...",Dialog menyampaikan curhat yang jelas dan emos...,4.62,7.438


In [51]:
# ============================================================
# LANGKAH 2: Load Model Sentence Embedding (HuggingFace)
# ============================================================
# Model: paraphrase-multilingual-MiniLM-L12-v2
# Model ini menghasilkan vektor embedding 384 dimensi yang
# merepresentasikan makna semantik dari teks (termasuk Bahasa Indonesia)
import torch
from transformers import AutoTokenizer, AutoModel

model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

print(f"Model '{model_name}' berhasil dimuat!")
print(f"Dimensi embedding: {model.config.hidden_size}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' berhasil dimuat!
Dimensi embedding: 384


In [52]:
# ============================================================
# LANGKAH 3: Fungsi untuk Membuat Sentence Embedding
# ============================================================
# Teknik: Mean Pooling — mengambil rata-rata vektor dari
# seluruh hidden states untuk menghasilkan satu vektor
# representasi semantik dari teks input
def get_embedding(texts):
    """
    Mengubah teks menjadi vektor embedding numerik.
    Input: list of strings
    Output: numpy array (n_texts, 384)
    """
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    # Mean pooling — rata-rata vektor dari hidden states
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.numpy()

# Contoh: embedding dari teks_kritik original (data asli dari dataset)
contoh_teks = df[df['jenis'] == 'original']['teks_kritik'].iloc[0]
contoh_embedding = get_embedding([contoh_teks])
print(f"Contoh embedding untuk teks_kritik original:")
print(f"  '{contoh_teks[:100]}...'")
print(f"  Shape: {contoh_embedding.shape}")
print(f"  5 nilai pertama: {contoh_embedding[0][:5]}")

Contoh embedding untuk teks_kritik original:
  'Dialog menyampaikan masalah yang jelas dan spesifik, yaitu konflik dengan pacar akibat membatalkan j...'
  Shape: (1, 384)
  5 nilai pertama: [ 0.16590638  0.14014664  0.0075111   0.12261109 -0.12160595]


In [53]:
# ============================================================
# LANGKAH 4: Hitung Cosine Similarity — Original vs Rephrase
# ============================================================
# Alur:
# 1. Ambil teks_kritik ORIGINAL (jenis='original') sbg acuan
# 2. Ambil semua teks_kritik REPHRASE (jenis='rephrase')
# 3. Generate embedding untuk original dan semua rephrase
# 4. Hitung cosine similarity antara embedding original dan
#    masing-masing embedding rephrase
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(dataframe):
    results_list = []
    summary_list = []
    
    for label_name in ['good', 'bad']:
        sub_df = dataframe[dataframe['label'] == label_name].copy()
        
        # Ambil teks kritik ORIGINAL sebagai acuan
        orig_row = sub_df[sub_df['jenis'] == 'original'].iloc[0]
        orig_text = orig_row['teks_kritik']
        
        # Ambil semua teks kritik REPHRASE
        rephrase_df = sub_df[sub_df['jenis'] == 'rephrase'].copy()
        rephrase_texts = rephrase_df['teks_kritik'].tolist()
        
        # Generate embedding
        orig_embedding = get_embedding([orig_text])
        rephrase_embeddings = get_embedding(rephrase_texts)
        
        # Hitung cosine similarity
        sim_scores = cosine_similarity(orig_embedding, rephrase_embeddings).flatten()
        
        # Simpan ke dataframe detail
        rephrase_df['cosine_similarity'] = sim_scores
        results_list.append(rephrase_df)
        
        # Simpan ringkasan per label
        summary_list.append({
            'label': label_name,
            'mean_cosine_similarity': np.mean(sim_scores),
            'std_cosine_similarity': np.std(sim_scores),
            'min_cosine_similarity': np.min(sim_scores),
            'max_cosine_similarity': np.max(sim_scores)
        })
    
    df_detailed = pd.concat(results_list, ignore_index=True)
    df_summary = pd.DataFrame(summary_list)
    
    return df_detailed, df_summary

# Jalankan fungsi
df_detailed, df_summary = calculate_cosine_similarity(df)
print("Perhitungan cosine similarity selesai!")
print(f"Total data hasil: {len(df_detailed)}")

Perhitungan cosine similarity selesai!
Total data hasil: 20


In [54]:
# ============================================================
# LANGKAH 5: Ringkasan Cosine Similarity per Label
# ============================================================
print("=== RINGKASAN COSINE SIMILARITY PER LABEL ===")
df_summary

=== RINGKASAN COSINE SIMILARITY PER LABEL ===


,label,mean_cosine_similarity,std_cosine_similarity,min_cosine_similarity,max_cosine_similarity
0,good,0.905276,0.017372,0.875022,0.936949
1,bad,0.868492,0.039026,0.803322,0.930188


In [55]:
# ============================================================
# LANGKAH 6: Statistik Detail per Label
# ============================================================
print("=== STATISTIK COSINE SIMILARITY PER LABEL ===\n")

# Filter hanya data rephrase
df_rephrase_cosine = df_detailed[['label', 'cosine_similarity']].copy()

for label in ['good', 'bad']:
    label_data = df_rephrase_cosine[df_rephrase_cosine['label'] == label]
    print(f"Label '{label}':")
    print(f"  Jumlah sampel      : {len(label_data)}")
    print(f"  Rata-rata (mean)   : {label_data['cosine_similarity'].mean():.4f}")
    print(f"  Standar deviasi    : {label_data['cosine_similarity'].std():.4f}")
    print(f"  Nilai minimum      : {label_data['cosine_similarity'].min():.4f}")
    print(f"  Nilai maksimum     : {label_data['cosine_similarity'].max():.4f}")
    print()

print("=== RINGKASAN KESELURUHAN ===")
print(f"  Total sampel       : {len(df_rephrase_cosine)}")
print(f"  Rata-rata (mean)   : {df_rephrase_cosine['cosine_similarity'].mean():.4f}")
print(f"  Standar deviasi    : {df_rephrase_cosine['cosine_similarity'].std():.4f}")

=== STATISTIK COSINE SIMILARITY PER LABEL ===

Label 'good':
  Jumlah sampel      : 10
  Rata-rata (mean)   : 0.9053
  Standar deviasi    : 0.0183
  Nilai minimum      : 0.8750
  Nilai maksimum     : 0.9369

Label 'bad':
  Jumlah sampel      : 10
  Rata-rata (mean)   : 0.8685
  Standar deviasi    : 0.0411
  Nilai minimum      : 0.8033
  Nilai maksimum     : 0.9302

=== RINGKASAN KESELURUHAN ===
  Total sampel       : 20
  Rata-rata (mean)   : 0.8869
  Standar deviasi    : 0.0363


In [56]:
# ============================================================
# LANGKAH 7: Tabel Lengkap Cosine Similarity (Semua Data)
# ============================================================
print("=== TABEL COSINE SIMILARITY — SEMUA DATA REPHRASE ===\n")
print(f"Total data: {len(df_rephrase_cosine)}\n")
display(df_rephrase_cosine)

=== TABEL COSINE SIMILARITY — SEMUA DATA REPHRASE ===

Total data: 20



,label,cosine_similarity
0,good,0.897816
1,good,0.886895
2,good,0.922240
3,good,0.875022
4,good,0.922214
5,good,0.936949
6,good,0.907745
7,good,0.907644
8,good,0.895198
9,good,0.901039


In [57]:
# ============================================================
# LANGKAH 8: Simpan Hasil ke CSV
# ============================================================
df_detailed.to_csv('hasil_cosine_similarity.csv', index=False)
print("Hasil detail berhasil disimpan ke 'hasil_cosine_similarity.csv'!")
print(f"Kolom tersimpan: {list(df_detailed.columns)}")

Hasil detail berhasil disimpan ke 'hasil_cosine_similarity.csv'!
Kolom tersimpan: ['label', 'jenis', 'teks_dialog', 'teks_kritik', 'skor_critic', 'waktu_detik', 'cosine_similarity']


In [58]:
# ============================================================
# TABEL ASAL — Good 3 & Bad 3 (Rephrase only)
# ============================================================
import pandas as pd

df_asal = pd.read_csv('output/tabel_perbandingan_lengkap_cosine.csv')

df_good = df_asal[(df_asal['label'] == 'good') & (df_asal['jenis'] == 'rephrase')].head(3)
df_bad = df_asal[(df_asal['label'] == 'bad') & (df_asal['jenis'] == 'rephrase')].head(3)
df_tabel = pd.concat([df_good, df_bad], ignore_index=True)

# Drop kolom teks_dialog karena isinya sama semua
df_tabel = df_tabel.drop(columns=['teks_dialog'])

display(df_tabel)

,label,jenis,teks_kritik,skor_critic,waktu_detik
0,good,rephrase,Dialog menyampaikan curhat secara jelas dan em...,4.50,7.614
1,good,rephrase,Dialog menyampaikan konflik yang jelas dan spe...,4.50,7.011
2,good,rephrase,Dialog menyampaikan konflik yang konkret dan s...,4.50,7.838
3,bad,rephrase,Dialog menghadirkan konflik yang spesifik dan ...,3.88,11.035
4,bad,rephrase,Konflik tentang lemari warisan disampaikan sec...,3.75,11.796
5,bad,rephrase,Konflik mengenai lemari warisan disampaikan se...,3.88,12.027


In [59]:
# ============================================================
# TABEL EMBEDDING VECTOR — Good 3 & Bad 3 (Original + Rephrase)
# ============================================================
import pandas as pd

df_asal = pd.read_csv('output/tabel_perbandingan_lengkap_cosine.csv')

# Ambil 3 good + 3 bad (masing-masing: 1 original + 2 rephrase)
df_good = df_asal[df_asal['label'] == 'good'].head(3)
df_bad = df_asal[df_asal['label'] == 'bad'].head(3)
df_sample = pd.concat([df_good, df_bad], ignore_index=True)

# Generate embedding untuk teks_kritik
teks_list = df_sample['teks_kritik'].tolist()
embeddings = get_embedding(teks_list)  # shape: (6, 384)

# Buat tabel: label, jenis, 5 dimensi pertama + shape info
rows = []
for i, row in df_sample.iterrows():
    vec = embeddings[i]
    rows.append({
        'label': row['label'],
        'jenis': row['jenis'],
        'skor_critic': row['skor_critic'],
        'd0': round(vec[0], 4),
        'd1': round(vec[1], 4),
        'd2': round(vec[2], 4),
        'd3': round(vec[3], 4),
        'd4': round(vec[4], 4),
        '...': '...',
        'd379': round(vec[379], 4),
        'd380': round(vec[380], 4),
        'd381': round(vec[381], 4),
        'd382': round(vec[382], 4),
        'd383': round(vec[383], 4),
    })

df_embed = pd.DataFrame(rows)
print(f"Shape embedding: {embeddings.shape} (6 teks × 384 dimensi)")
print(f"Range nilai: [{embeddings.min():.4f}, {embeddings.max():.4f}]\n")
display(df_embed)

Shape embedding: (6, 384) (6 teks × 384 dimensi)
Range nilai: [-0.4806, 0.4656]



,label,jenis,skor_critic,d0,d1,d2,d3,d4,...,d379,d380,d381,d382,d383
0,good,original,4.88,0.1574,0.1358,0.0219,0.1246,-0.0966,...,0.1331,0.1955,0.3794,-0.0085,-0.0489
1,good,rephrase,4.50,0.2717,0.0996,-0.1027,0.1687,-0.1078,...,0.2045,0.1842,0.3510,0.0108,-0.0813
2,good,rephrase,4.50,0.1098,0.1098,0.0423,0.2490,-0.0238,...,0.0462,0.2820,0.3440,0.0213,-0.0132
3,bad,original,3.75,0.1956,0.1564,0.0117,0.3230,-0.0362,...,0.1475,0.4607,0.1978,0.0951,-0.0486
4,bad,rephrase,3.88,0.1905,0.1114,-0.0298,0.4114,-0.0255,...,0.1883,0.3011,0.2379,0.1159,-0.0498
5,bad,rephrase,3.75,0.0848,0.2254,0.0234,0.2958,-0.0630,...,0.1479,0.3935,0.1916,0.0573,-0.0854


In [60]:
# ============================================================
# HASIL COSINE SIMILARITY — Good 3 & Bad 3
# ============================================================
df_good_cos = df_detailed[df_detailed['label'] == 'good'].head(3)
df_bad_cos = df_detailed[df_detailed['label'] == 'bad'].head(3)
df_cos = pd.concat([df_good_cos, df_bad_cos], ignore_index=True)

display(df_cos[['label', 'jenis', 'skor_critic', 'cosine_similarity']])

,label,jenis,skor_critic,cosine_similarity
0,good,rephrase,4.50,0.897816
1,good,rephrase,4.50,0.886895
2,good,rephrase,4.50,0.922240
3,bad,rephrase,3.88,0.854948
4,bad,rephrase,3.75,0.883234
5,bad,rephrase,3.88,0.904939
